# 05. B-cell stage confound

All 10 ALL1/AF4 patients are in B-cell stage B1 (notebook 01). A classifier could then separate ALL1/AF4 by recognising the cell stage rather than the genetic subtype. Three checks:

1. Class by stage table.
2. The best baseline workflow (`s2_knn3_z`, plus `s1_knn3_z` for comparison) on stage-B1 patients only. Within one stage, stage cannot explain any separation.
3. How well the stage itself can be predicted from the genes, with the same pipeline.

Resampling is the same as in the baseline (100 rounds, 30% hold-out, stratified, seed 0, selection inside every round).

In [1]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
from pipeline import evaluate, load_data, make_models

X, y = load_data()
stage = pd.read_csv('../data/labels.csv', index_col='patient', dtype={'patient': str})['BT']
knn = {'knn3_z': make_models()['knn3_z']}

pd.crosstab(stage, y)

class,ALL1/AF4,BCR/ABL,E2A/PBX1,NEG
BT,,,,
B,0,2,1,2
B1,10,1,0,8
B2,0,19,0,16
B3,0,8,1,14
B4,0,7,3,2


## Within stage B1 only

In [2]:
b1 = stage == 'B1'
print(y[b1].value_counts().to_dict())

# one BCR/ABL patient cannot be split across train and test, so it is left out
keep = b1 & (y != 'BCR/ABL')
X1, y1 = X[keep], y[keep]
print('patients used:', len(y1), '| reference accuracy of the larger class:', round(y1.value_counts(normalize=True).max(), 3))

within, _ = evaluate(X1, y1, n_rounds=100, seed=0, models=knn, classes=['ALL1/AF4', 'NEG'])
within.to_csv('../results/b1_within_stage_scores.csv', index=False)
within.groupby('selection')[['accuracy', 'balanced_accuracy', 'ALL1/AF4', 'NEG']].mean().round(3)

{'ALL1/AF4': 10, 'NEG': 8, 'BCR/ABL': 1}
patients used: 18 | reference accuracy of the larger class: 0.556


,accuracy,balanced_accuracy,ALL1/AF4,NEG
selection,,,,
s1,0.932,0.932,0.967,0.897
s2,0.960,0.960,0.977,0.943


## Predicting the B-cell stage from the genes

In [3]:
stages = sorted(stage.unique())
stage5, _ = evaluate(X, stage, n_rounds=100, seed=0, models=knn, classes=stages)
print('five stages | majority-stage accuracy:', round(stage.value_counts(normalize=True).max(), 3), '| chance balanced accuracy: 0.2')
print(stage5.groupby('selection')[['accuracy', 'balanced_accuracy'] + stages].mean().round(3))

b1_vs_rest = pd.Series(np.where(stage == 'B1', 'B1', 'other'), index=stage.index)
binary, _ = evaluate(X, b1_vs_rest, n_rounds=100, seed=0, models=knn, classes=['B1', 'other'])
print()
print('B1 vs other | majority accuracy:', round(b1_vs_rest.value_counts(normalize=True).max(), 3), '| chance balanced accuracy: 0.5')
print(binary.groupby('selection')[['accuracy', 'balanced_accuracy', 'B1', 'other']].mean().round(3))
pd.concat({'five_stages': stage5, 'b1_vs_other': binary}).to_csv('../results/stage_prediction_scores.csv')

five stages | majority-stage accuracy: 0.372 | chance balanced accuracy: 0.2
           accuracy  balanced_accuracy     B     B1     B2     B3     B4
selection                                                               
s1            0.527              0.416  0.22  0.573  0.757  0.393  0.135
s2            0.535              0.420  0.17  0.597  0.735  0.431  0.168



B1 vs other | majority accuracy: 0.798 | chance balanced accuracy: 0.5
           accuracy  balanced_accuracy     B1  other
selection                                           
s1            0.887              0.768  0.563  0.972
s2            0.892              0.774  0.573  0.975


## Notes

**Result**

- Within stage B1 only (10 ALL1/AF4 and 8 NEG patients), the workflow separates the two classes with accuracy 0.96 and balanced accuracy 0.96 (`s2_knn3_z`; 0.93 with all ANOVA genes). The larger class is only 55.6% of these patients. Stage is constant here, so it cannot explain the separation.
- The stage itself is only partly encoded in the genes. Predicting the five stages gives accuracy 0.53 (majority stage: 0.37) and balanced accuracy 0.42 (chance: 0.20). B1 against all other stages gives balanced accuracy 0.77 (chance: 0.50), with only 57% of B1 patients recognised.

**What it means**

- The model does separate ALL1/AF4 from NEG by something other than the stage. The claim that the classifier finds subtype signal is supported for this class pair.
- Stage is not irrelevant: B1 is recognised better than chance. Part of the very high ALL1/AF4 recall in the baseline (0.997 for `s2_knn3_z`) may still come from stage information, but the within-stage accuracy of 0.96 shows that this cannot be all of it.

**Limits**

- Only 18 patients are used in the within-stage run, and 30% hold-out tests about 5 or 6 of them per round, so each round is very coarse. Averages over 100 rounds are reasonable but depend on a small sample; the rounds share training patients.
- The single BCR/ABL patient in stage B1 is left out (one patient cannot be stratified), so the check covers ALL1/AF4 against NEG only. E2A/PBX1 (5 patients, none in B1) and BCR/ABL are not tested for a stage confound.
- Eight NEG patients may differ from the ALL1/AF4 patients in ways other than stage (for example age or sample handling); this check removes the stage explanation only.